# 1. ARCADE Data Preparation for Vessel Segmentation

Load ARCADE dataset and prepare for FPN + U-Net + Swin-Transformer training

In [ ]:
import json
import os
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
from sklearn.model_selection import KFold
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

print("✓ All imports successful")

## Setup Paths

In [ ]:
# Configure paths
ARCADE_ROOT = Path('D:/MPHIL_CODES/MPHIL_MAIN_REPO/experiments/paper_implementations/cGAN/ARCADE')
OUTPUT_DIR = Path('D:/MPHIL_CODES/MPHIL_MAIN_REPO/experiments/paper_implementations/vessel_segmentation/data')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Dataset: {ARCADE_ROOT}")
print(f"Output: {OUTPUT_DIR}")
print(f"\n✓ ARCADE exists: {ARCADE_ROOT.exists()}")

if ARCADE_ROOT.exists():
    print("\nARCADE structure:")
    for item in ARCADE_ROOT.iterdir():
        print(f"  {item.name}")

## Load ARCADE Annotations

In [ ]:
# Load train annotations
train_ann_file = ARCADE_ROOT / 'stenosis' / 'train' / 'annotations' / 'train.json'
val_ann_file = ARCADE_ROOT / 'stenosis' / 'val' / 'annotations' / 'val.json'

with open(train_ann_file, 'r') as f:
    train_coco = json.load(f)

with open(val_ann_file, 'r') as f:
    val_coco = json.load(f)

print(f"Train images: {len(train_coco['images'])}")
print(f"Train annotations: {len(train_coco['annotations'])}")
print(f"Train categories: {len(train_coco['categories'])}")

print(f"\nVal images: {len(val_coco['images'])}")
print(f"Val annotations: {len(val_coco['annotations'])}")
print(f"Val categories: {len(val_coco['categories'])}")

## Create Binary Vessel Masks

In [ ]:
def create_binary_masks(coco_data, image_dir, mask_dir):
    """Create binary vessel segmentation masks from COCO annotations."""
    mask_dir.mkdir(parents=True, exist_ok=True)
    
    images_dict = {im["id"]: im for im in coco_data["images"]}
    anns_by_img = defaultdict(list)
    
    for ann in coco_data["annotations"]:
        anns_by_img[ann["image_id"]].append(ann)
    
    created_count = 0
    
    for img_id, anns in anns_by_img.items():
        info = images_dict[img_id]
        w, h = info['width'], info['height']
        
        # Create binary mask
        binary_mask = Image.new('L', (w, h), 0)
        draw = ImageDraw.Draw(binary_mask)
        
        # Draw all annotations (vessels)
        for ann in anns:
            if 'segmentation' in ann and ann['segmentation']:
                for seg in ann['segmentation']:
                    if len(seg) >= 6:
                        pts = [(seg[i], seg[i+1]) for i in range(0, len(seg), 2)]
                        draw.polygon(pts, fill=255)
        
        # Save mask
        mask_name = Path(info['file_name']).stem + '.png'
        mask_path = mask_dir / mask_name
        binary_mask.save(mask_path)
        created_count += 1
    
    return created_count

# Create masks for train and val
train_mask_dir = OUTPUT_DIR / 'train' / 'masks'
val_mask_dir = OUTPUT_DIR / 'val' / 'masks'

train_image_dir = ARCADE_ROOT / 'stenosis' / 'train' / 'images'
val_image_dir = ARCADE_ROOT / 'stenosis' / 'val' / 'images'

print("Creating binary vessel masks...")
train_count = create_binary_masks(train_coco, train_image_dir, train_mask_dir)
val_count = create_binary_masks(val_coco, val_image_dir, val_mask_dir)

print(f"✓ Created {train_count} training masks")
print(f"✓ Created {val_count} validation masks")

## Create 5-Fold Cross-Validation Split

In [ ]:
# Combine train and val for 5-fold CV
all_images = train_coco['images'] + val_coco['images']
image_ids = [img['id'] for img in all_images]
file_names = [img['file_name'] for img in all_images]

print(f"Total images for 5-fold CV: {len(all_images)}")

# Create 5-fold split
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

fold_splits = []
for fold_idx, (train_idx, val_idx) in enumerate(kfold.split(all_images)):
    fold_splits.append({
        'fold': fold_idx,
        'train_indices': train_idx,
        'val_indices': val_idx,
        'train_images': [image_ids[i] for i in train_idx],
        'val_images': [image_ids[i] for i in val_idx]
    })
    print(f"Fold {fold_idx}: Train={len(train_idx)}, Val={len(val_idx)}")

# Save fold split info
fold_info = {
    'all_images': image_ids,
    'folds': fold_splits
}

with open(OUTPUT_DIR / 'fold_split.json', 'w') as f:
    # Convert indices to lists for JSON serialization
    fold_data = []
    for fold in fold_splits:
        fold_data.append({
            'fold': fold['fold'],
            'train_indices': fold['train_indices'].tolist(),
            'val_indices': fold['val_indices'].tolist(),
            'train_count': len(fold['train_indices']),
            'val_count': len(fold['val_indices'])
        })
    json.dump(fold_data, f, indent=2)

print("\n✓ Saved fold split info")

## Create Dataset Index

In [ ]:
# Create a combined index of all images with their paths
dataset_index = []

# Add training images
for img in train_coco['images']:
    dataset_index.append({
        'image_id': img['id'],
        'file_name': img['file_name'],
        'image_path': str(train_image_dir / img['file_name']),
        'mask_path': str(train_mask_dir / (Path(img['file_name']).stem + '.png')),
        'split': 'train_original',
        'width': img['width'],
        'height': img['height']
    })

# Add validation images
for img in val_coco['images']:
    dataset_index.append({
        'image_id': img['id'],
        'file_name': img['file_name'],
        'image_path': str(val_image_dir / img['file_name']),
        'mask_path': str(val_mask_dir / (Path(img['file_name']).stem + '.png')),
        'split': 'val_original',
        'width': img['width'],
        'height': img['height']
    })

# Save dataset index
df_index = pd.DataFrame(dataset_index)
df_index.to_csv(OUTPUT_DIR / 'dataset_index.csv', index=False)

print(f"Dataset index:")
print(df_index.head())
print(f"\nTotal: {len(df_index)} images")
print(f"Train (original): {(df_index['split']=='train_original').sum()}")
print(f"Val (original): {(df_index['split']=='val_original').sum()}")

## Visualize Sample Data

In [ ]:
# Show a few samples
fig, axes = plt.subplots(3, 2, figsize=(10, 12))

for idx in range(3):
    sample = dataset_index[idx]
    
    # Load image
    img = Image.open(sample['image_path']).convert('L')
    img_array = np.array(img)
    
    # Load mask
    mask = Image.open(sample['mask_path']).convert('L')
    mask_array = np.array(mask)
    
    # Plot image
    axes[idx, 0].imshow(img_array, cmap='gray')
    axes[idx, 0].set_title(f"Image {idx+1}\n{sample['file_name']}")
    axes[idx, 0].axis('off')
    
    # Plot mask
    axes[idx, 1].imshow(mask_array, cmap='gray')
    axes[idx, 1].set_title(f"Mask {idx+1}\nVessel Segmentation")
    axes[idx, 1].axis('off')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'sample_data.png', dpi=150, bbox_inches='tight')
print("✓ Saved sample visualization")
plt.show()

## Dataset Statistics

In [ ]:
print("\n" + "="*60)
print("DATASET PREPARATION COMPLETE")
print("="*60)

print(f"\nTotal images: {len(dataset_index)}")
print(f"Original train: {(df_index['split']=='train_original').sum()}")
print(f"Original val: {(df_index['split']=='val_original').sum()}")

print(f"\n5-Fold Cross-Validation Setup:")
for fold in fold_splits:
    print(f"  Fold {fold['fold']}: {len(fold['train_indices'])} train, {len(fold['val_indices'])} val")

print(f"\nOutputs:")
print(f"  - Dataset index: {OUTPUT_DIR / 'dataset_index.csv'}")
print(f"  - Fold splits: {OUTPUT_DIR / 'fold_split.json'}")
print(f"  - Training masks: {train_mask_dir}")
print(f"  - Validation masks: {val_mask_dir}")

print(f"\n✓ Ready for model training!")